In [ ]:
# set up

In [1]:
!pip -q install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip -q install --upgrade transformers datasets evaluate accelerate kagglehub scikit-learn


In [ ]:
# Dataset loader

In [7]:
import pandas as pd
from pathlib import Path
from datasets import Dataset, load_dataset, concatenate_datasets

ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABEL2ID = {v:k for k,v in ID2LABEL.items()}

def load_financial_news_kagglehub():
    """Kaggle: ankurzing/sentiment-analysis-for-financial-news (3-class)."""
    import kagglehub
    path = kagglehub.dataset_download("ankurzing/sentiment-analysis-for-financial-news")
    df = pd.read_csv(
        Path(path) / "all-data.csv",
        encoding="ISO-8859-1",
        names=["label","text"]
    )
    df["label"] = df["label"].str.lower().map(LABEL2ID)
    df = df.dropna().drop_duplicates("text")
    return Dataset.from_pandas(df[["text","label"]])

def load_cosmos98_twitter_reddit_kagglehub():
    """
    Kaggle: cosmos98/twitter-and-reddit-sentimental-analysis-dataset.
    Walks all CSVs, finds text/label columns, normalizes labels to {0,1,2}.
    """
    import kagglehub
    root = Path(kagglehub.dataset_download("cosmos98/twitter-and-reddit-sentimental-analysis-dataset"))
    frames = []
    for f in root.rglob("*.csv"):
        try:
            df = pd.read_csv(f)
        except Exception:
            continue
        cols = {c.lower(): c for c in df.columns}
        tcol = next((cols[k] for k in ("text","content","clean_text","tweet","body","sentence") if k in cols), None)
        lcol = next((cols[k] for k in ("sentiment","label","polarity","target","category") if k in cols), None)
        if not tcol or not lcol:
            continue
        df = df[[tcol, lcol]].rename(columns={tcol:"text", lcol:"label"})

        if df["label"].dtype == "O":
            df["label"] = df["label"].astype(str).str.lower().map({"negative":0,"neutral":1,"positive":2})
        else:
            uniq = set(pd.Series(df["label"]).dropna().unique().tolist())
            if uniq <= {-1,0,1}:
                df["label"] = df["label"].map({-1:0,0:1,1:2})
            # if already {0,1,2}, keep as is

        frames.append(df.dropna())

    df_all = pd.concat(frames, ignore_index=True).drop_duplicates("text")
    print("cosmos98 size:", len(df_all), " label counts:", df_all["label"].value_counts().to_dict())
    return Dataset.from_pandas(df_all[["text","label"]])

def load_sst2_as_three_class():
    """GLUE SST-2 (binary) mapped into 3-class space (neutral unused)."""
    ds = load_dataset("glue", "sst2")
    def to_ds(split):
        df = pd.DataFrame({"text": ds[split]["sentence"], "label": ds[split]["label"]})
        df["label"] = df["label"].map({0:0, 1:2})  # 0->neg, 1->pos ; leave 1(neutral) unused
        return Dataset.from_pandas(df[["text","label"]])
    return {"train": to_ds("train"), "validation": to_ds("validation")}


In [ ]:
# Build training/eval sets

In [11]:
from datasets import ClassLabel, concatenate_datasets

# 1) Load
fin_ds    = load_financial_news_kagglehub()
cosmos_ds = load_cosmos98_twitter_reddit_kagglehub()
sst2      = load_sst2_as_three_class()  # dict with "train"/"validation"

# 2) Cast labels to ClassLabel (required for stratify_by_column)
cls = ClassLabel(names=["negative","neutral","positive"])
fin_ds    = fin_ds.cast_column("label", cls)
cosmos_ds = cosmos_ds.cast_column("label", cls)
sst2_train = sst2["train"].cast_column("label", cls)
sst2_val   = sst2["validation"].cast_column("label", cls)

# 3) Now you can stratified-split the HF datasets
fin_split    = fin_ds.train_test_split(test_size=0.2, seed=CFG.seed, stratify_by_column="label")
cosmos_split = cosmos_ds.train_test_split(test_size=0.2, seed=CFG.seed, stratify_by_column="label")

# 4) Combine (adjust ratios as you like)
train_ds = concatenate_datasets([
    fin_split["train"],
    cosmos_split["train"],
    sst2_train.shuffle(seed=CFG.seed).select(range(30000))  # optional subsample
])
eval_ds  = concatenate_datasets([
    fin_split["test"],
    cosmos_split["test"],
    sst2_val
])

print(train_ds, eval_ds)


cosmos98 size: 162969  label counts: {2.0: 72249, 1.0: 55211, 0.0: 35509}


Casting the dataset:   0%|          | 0/4838 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/162969 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/67349 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/872 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 164245
}) Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 34434
})


In [ ]:
# Tokenize

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(CFG.ckpt_dir if CFG.ckpt_dir.exists() else CFG.base_model),
    use_fast=True
)

def tok(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=CFG.max_len)

tok_train = train_ds.map(tok, batched=True, remove_columns=[c for c in train_ds.column_names if c not in ["text","label"]])
tok_eval  = eval_ds.map(tok,  batched=True, remove_columns=[c for c in eval_ds.column_names  if c not in ["text","label"]])

tok_train = tok_train.rename_column("label","labels").with_format("torch")
tok_eval  = tok_eval.rename_column("label","labels").with_format("torch")


Map:   0%|          | 0/164245 [00:00<?, ? examples/s]

Map:   0%|          | 0/34434 [00:00<?, ? examples/s]

In [ ]:
# Continue training from your checkpoint

In [17]:
# === Load saved checkpoint and tokenizer ===
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import numpy as np, evaluate

CKPT_DIR = Path("./deberta-financial")   # your previously saved model
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABEL2ID = {v:k for k,v in ID2LABEL.items()}

tokenizer = AutoTokenizer.from_pretrained(str(CKPT_DIR), use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    str(CKPT_DIR),
    num_labels=3,
    id2label=ID2LABEL, label2id=LABEL2ID
)

# === If you already built tok_train/tok_eval for NEW data, just plug them here ===
# tok_train / tok_eval must have columns: input_ids, attention_mask, labels (as ints 0/1/2)

acc = evaluate.load("accuracy"); f1 = evaluate.load("f1")
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=p.label_ids, average="macro")["f1"]
    }

args = TrainingArguments(
    output_dir="out-continued",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="steps",        # transformers v5 name
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,   # <= your new tokenized train set
    eval_dataset=tok_eval,     # <= your new tokenized eval set
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Continue fine-tuning on the NEW data (no full retrain needed)
trainer.train()
model.save_pretrained(str(CKPT_DIR))
tokenizer.save_pretrained(str(CKPT_DIR))


/var/folders/ng/prfj2ls9255881q552hlmp680000gn/T/ipykernel_91136/3457166873.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
500,0.721400,0.509160,0.809113,0.791723
1000,0.475800,0.414551,0.863565,0.849321
1500,0.376900,0.354772,0.878899,0.868700
2000,0.332000,0.325422,0.898995,0.888515
2500,0.303300,0.274736,0.918482,0.911193
3000,0.280900,0.270939,0.917059,0.910729
3500,0.260900,0.236441,0.938026,0.932451
4000,0.239900,0.208530,0.942876,0.937771
4500,0.232900,0.208293,0.944909,0.939975
5000,0.231100,0.212224,0.939595,0.934139


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argume

('deberta-financial/tokenizer_config.json',
 'deberta-financial/special_tokens_map.json',
 'deberta-financial/spm.model',
 'deberta-financial/added_tokens.json',
 'deberta-financial/tokenizer.json')

In [ ]:
# Evaluate on the validation set

In [19]:
metrics = trainer.evaluate(tok_eval)
print(metrics)


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.14761453866958618, 'eval_accuracy': 0.9574838822094441, 'eval_f1_macro': 0.9536420394893446, 'eval_runtime': 406.8257, 'eval_samples_per_second': 84.641, 'eval_steps_per_second': 2.647, 'epoch': 0.8766803039158387}


In [ ]:
# works better than before, but may still need extra work with the sarcastic side

In [25]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis", model="./deberta-financial", tokenizer="./deberta-financial")

post1 = "SLA went green lmao."
post2 = "385$ Calls? That wont even happen if Elon promised flying cars in Q2"

print(sentiment_pipeline([post1, post2]))


Device set to use mps:0


[{'label': 'positive', 'score': 0.734401524066925}, {'label': 'neutral', 'score': 0.8226130604743958}]


In [ ]:
# ticker extraction function

In [27]:
import re

def extract_tickers(text):
    return re.findall(r'\b[A-Z]{2,5}\b', text)

In [29]:
extract_tickers("SLA went green lmao.")

['SLA']

In [ ]:
# extra trainning with the sarcasm detection
# first get the dataset

In [33]:
import kagglehub
path = kagglehub.dataset_download("danofer/sarcasm")
print("Kaggle path:", path)

100%|████████████████████████████████████████| 216M/216M [00:02<00:00, 85.3MB/s]

Extracting files...


Kaggle path: /Users/rickliu/.cache/kagglehub/datasets/danofer/sarcasm/versions/4


In [ ]:
# loader

In [35]:
# expects `path` from kagglehub.dataset_download("danofer/sarcasm")

from pathlib import Path
import pandas as pd
from datasets import Dataset, ClassLabel

def load_danofer_sarcasm(path_root: str | Path) -> Dataset:
    """
    Loads Kaggle 'danofer/sarcasm' dataset.
    Tries typical files (e.g., train-balanced-sarcasm.csv) and column names.
    Normalizes to columns: text (str), label (0=not sarcastic, 1=sarcastic).
    """
    root = Path(path_root)
    frames = []

    # collect csvs (common main file: 'train-balanced-sarcasm.csv')
    for f in root.rglob("*.csv"):
        try:
            df = pd.read_csv(f)
        except Exception:
            continue

        cols = {c.lower(): c for c in df.columns}
        text_col = next((cols[k] for k in ("text","comment","headline","utterance","content") if k in cols), None)
        label_col = next((cols[k] for k in ("label","sarcasm","is_sarcastic","class") if k in cols), None)
        if not text_col or not label_col:
            continue

        df = df[[text_col, label_col]].rename(columns={text_col:"text", label_col:"label"})
        # normalize labels to {0,1}
        if df["label"].dtype == "O":
            df["label"] = df["label"].astype(str).str.lower().map({"0":0,"1":1,"false":0,"true":1,"not sarcastic":0,"sarcastic":1})
        df = df.dropna().drop_duplicates("text")
        frames.append(df)

    if not frames:
        raise RuntimeError("Could not find usable CSVs in the danofer/sarcasm folder.")

    df_all = pd.concat(frames, ignore_index=True)
    print("danofer/sarcasm rows:", len(df_all), " label counts:", df_all["label"].value_counts().to_dict())

    # ensure ints 0/1
    df_all["label"] = df_all["label"].astype(int).clip(0, 1)
    ds = Dataset.from_pandas(df_all[["text","label"]], preserve_index=False)

    # cast to ClassLabel for stratified splits later
    ds = ds.cast_column("label", ClassLabel(names=["not sarcastic","sarcastic"]))
    return ds

danofer_ds = load_danofer_sarcasm(path)

danofer/sarcasm rows: 962293  label counts: {1: 484099, 0: 478194}


Casting the dataset:   0%|          | 0/962293 [00:00<?, ? examples/s]

In [ ]:
# Split (stratified), tokenize with your existing model’s tokenizer

In [37]:
from datasets import concatenate_datasets
from transformers import AutoTokenizer

# stratified split
danofer_split = danofer_ds.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_ds = danofer_split["train"]
eval_ds  = danofer_split["test"]

# use tokenizer from your previous sarcasm checkpoint
sarcasm_tokenizer = AutoTokenizer.from_pretrained("./sarcasm_detector", use_fast=True)
MAX_LEN_SARC = 128

def tok(batch):
    return sarcasm_tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN_SARC)

tok_train = train_ds.map(tok, batched=True, remove_columns=[c for c in train_ds.column_names if c not in ["text","label"]]).rename_column("label","labels").with_format("torch")
tok_eval  = eval_ds.map(tok,  batched=True, remove_columns=[c for c in eval_ds.column_names  if c not in ["text","label"]]).rename_column("label","labels").with_format("torch")


Map:   0%|          | 0/769834 [00:00<?, ? examples/s]

Map:   0%|          | 0/192459 [00:00<?, ? examples/s]

In [ ]:
# Continue fine-tuning from your previous model

In [ ]:
import torch, numpy as np, evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

sarcasm_model = AutoModelForSequenceClassification.from_pretrained("./sarcasm_detector", num_labels=2).to(DEVICE)
sarcasm_model.gradient_checkpointing_enable()  # cooler on Mac

acc = evaluate.load("accuracy"); f1 = evaluate.load("f1")
def compute_metrics(p):
    y_pred = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": acc.compute(predictions=y_pred, references=p.label_ids)["accuracy"],
        "f1_macro": f1.compute(predictions=y_pred, references=p.label_ids, average="macro")["f1"],
        "f1_sarcastic": f1.compute(predictions=y_pred, references=p.label_ids, average=None)["f1"][1]
    }

args = TrainingArguments(
    output_dir="out-sarcasm-continued",
    learning_rate=2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    dataloader_pin_memory=False,   # silence MPS warning
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    seed=42,
)

sarcasm_trainer = Trainer(
    model=sarcasm_model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_eval,
    tokenizer=sarcasm_tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

sarcasm_trainer.train()
sarcasm_model.save_pretrained("./sarcasm_detector")
sarcasm_tokenizer.save_pretrained("./sarcasm_detector")


/var/folders/ng/prfj2ls9255881q552hlmp680000gn/T/ipykernel_91136/2177163579.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  sarcasm_trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Sarcastic
500,0.673100,0.589973,0.688489,0.684515,0.649104
1000,0.589800,0.562526,0.706384,0.706006,0.695456
1500,0.580000,0.561007,0.712282,0.710690,0.689235
2000,0.572100,0.559464,0.711300,0.706506,0.668998
2500,0.562800,0.548642,0.721432,0.721379,0.717560
